In [1]:
from pathlib import Path
import csv

# 1. Setup paths
folder_path = Path.home() / "Documents" / "Activity_5_Files"
files = list(folder_path.glob("*.txt"))

results = []

# 2. Process each file
for file_path in files:
    content = file_path.read_text()
    
    # Cleaning data: ignoring empty lines
    lines = [line for line in content.splitlines() if line.strip()]
    text_data = " ".join(lines)
    words = text_data.split()
    characters = len(text_data.replace(" ", "")) # Chars excluding spaces
    
    # Calculate counts
    num_lines = len(lines)
    num_words = len(words)
    num_chars = characters
    
    # Calculate metrics (avoiding division by zero)
    wpl = num_words / num_lines if num_lines > 0 else 0
    cpw = num_chars / num_words if num_words > 0 else 0
    
    results.append({
        "Filename": file_path.name,
        "Lines": num_lines,
        "Words": num_words,
        "Characters": num_chars,
        "Words/Line": round(wpl, 2),
        "Chars/Word": round(cpw, 2)
    })

# 3. Sort by Word Density (Words/Line) - Highest to Lowest
results.sort(key=lambda x: x["Words/Line"], reverse=True)

# 4. Display Tabular Summary
print(f"{'Filename':<30} | {'Lines':<5} | {'Words':<5} | {'Chars':<6} | {'W/L':<6} | {'C/W':<6}")
print("-" * 80)
for r in results:
    print(f"{r['Filename']:<30} | {r['Lines']:<5} | {r['Words']:<5} | {r['Characters']:<6} | {r['Words/Line']:<6} | {r['Chars/Word']:<6}")

# 5. Identify Most and Least Dense
if results:
    print(f"\nMost Content-Dense File: {results[0]['Filename']}")
    print(f"Least Content-Dense File: {results[-1]['Filename']}")

# 6. Optional: Export to CSV
csv_path = folder_path / "content_summary.csv"
with open(csv_path, mode='w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

print(f"\nSummary exported to: {csv_path}")

Filename                       | Lines | Words | Chars  | W/L    | C/W   
--------------------------------------------------------------------------------
intro_TUPM-24-2017.txt         | 1     | 11    | 65     | 11.0   | 5.91  
merged_TUPM-24-2017.txt        | 4     | 17    | 80     | 4.25   | 4.71  
lines_TUPM-24-2017.txt         | 3     | 6     | 15     | 2.0    | 2.5   

Most Content-Dense File: intro_TUPM-24-2017.txt
Least Content-Dense File: lines_TUPM-24-2017.txt

Summary exported to: C:\Users\Reno\Documents\Activity_5_Files\content_summary.csv


In [2]:
from pathlib import Path
import csv
import string

# 1. Configuration
student_id = "TUPM-24-2017"
folder_path = Path.home() / "Documents" / "Activity_5_Files"
stop_words_file = folder_path / "stopwords.txt"

# Default stopwords if the file doesn't exist
stop_words = {"the", "is", "and", "to", "in", "it", "of", "a", "with", "for", "on"}

# Load stopwords from file if it exists
if stop_words_file.exists():
    custom_stops = stop_words_file.read_text().lower().split()
    stop_words.update(custom_stops)

# 2. Data Structures
overall_frequencies = {}
file_specific_data = {}

# 3. Processing Files
files = list(folder_path.glob("*.txt"))

for file_path in files:
    # Skip the stopwords file itself
    if file_path.name == "stopwords.txt":
        continue
        
    content = file_path.read_text(encoding='utf-8').lower()
    
    # Remove punctuation
    content = content.translate(str.maketrans('', '', string.punctuation))
    
    words = content.split()
    file_freq = {}
    
    for word in words:
        if word not in stop_words:
            # Update file-specific count
            file_freq[word] = file_freq.get(word, 0) + 1
            # Update overall count
            overall_frequencies[word] = overall_frequencies.get(word, 0) + 1
            
    file_specific_data[file_path.name] = file_freq

# 4. Output: Most Frequent Meaningful Words
sorted_overall = sorted(overall_frequencies.items(), key=lambda x: x[1], reverse=True)

print(f"--- Top 10 Meaningful Words Across All Files ---")
for word, count in sorted_overall[:10]:
    print(f"{word:<15}: {count} times")

# 5. Export results to CSV
csv_path = folder_path / f"word_frequency_{student_id}.csv"
with open(csv_path, mode='w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(["Word", "Overall_Frequency"])
    writer.writerows(sorted_overall)

print(f"\nFull frequency data exported to: {csv_path.name}")

--- Top 10 Meaningful Words Across All Files ---
line           : 6 times
welcome        : 2 times
arjames        : 2 times
e              : 2 times
kinilitan      : 2 times
id             : 2 times
tupm242017     : 2 times
file           : 2 times
handling       : 2 times
python         : 2 times

Full frequency data exported to: word_frequency_TUPM-24-2017.csv


In [3]:
from pathlib import Path
import shutil
from datetime import datetime

# 1. Student Configuration
student_id = "TUPM-24-2017"
student_name = "Arjames E. Kinilitan"

# 2. Path Setup
base_dir = Path.home() / "Documents" / "Activity_5_Files"
backup_dir = base_dir / f"backup_{student_id}"
log_file = backup_dir / f"backup_log_{student_id}.txt"

# Ensure the backup directory exists
backup_dir.mkdir(parents=True, exist_ok=True)

def perform_backup(filename):
    original_file = base_dir / filename
    
    # Check if the target file actually exists
    if not original_file.exists():
        print(f"Error: The file {filename} was not found in {base_dir}")
        return

    # 3. Generate Timestamp and Unique Filename
    # Format: YYYYMMDD_HHMMSS
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_filename = f"{original_file.stem}_{student_id}_{timestamp}{original_file.suffix}"
    destination_path = backup_dir / backup_filename

    # 4. Copy the File
    shutil.copy2(original_file, destination_path)
    file_size = destination_path.stat().st_size

    # 5. Update the Log File (Append Mode)
    log_entry = (
        f"{'='*50}\n"
        f"Backup Date/Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
        f"Student:          {student_name} ({student_id})\n"
        f"Original File:    {filename}\n"
        f"Backup File:      {backup_filename}\n"
        f"Size:             {file_size} bytes\n"
        f"Path:             {destination_path}\n"
    )

    with open(log_file, "a") as log:
        log.write(log_entry)

    # 6. Output Confirmation
    print(f"Backup completed for {student_id} ({student_name})")
    print(f"File saved as:  {backup_filename}")
    print(f"Log updated at: {backup_dir}")

# --- Execute Backup ---
# Replace 'intro_TUPM-24-2017.txt' with the file you want to back up
perform_backup(f"intro_{student_id}.txt")

Backup completed for TUPM-24-2017 (Arjames E. Kinilitan)
File saved as:  intro_TUPM-24-2017_TUPM-24-2017_20260417_145525.txt
Log updated at: C:\Users\Reno\Documents\Activity_5_Files\backup_TUPM-24-2017
